# 🧪 Overfitting, and choosing a model

In Lesson 1 you trained one model with one setting (`max_depth=4`) and took its score at face value. Professionals don't. This lesson answers two questions:

1. **How complex should a model be?** Too simple and it misses the pattern; too complex and it memorises noise.
2. **Which model should you deploy?** The obvious answer — *the most accurate one* — turns out to be an ethical choice in disguise.

Same data and same split as Lesson 1, so the results are comparable.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

df = pd.read_csv("data/student_digital_skills.csv")
features = ["age", "hours_online_learning", "digital_skill_score", "prior_ict_courses"]
X, y = df[features], df["passed_certification"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

baseline = max(y_test.mean(), 1 - y_test.mean())
print(f"Always predicting 'pass' would score {baseline:.0%} on the test set.")

## Overfitting: memorising instead of learning

A decision tree with more depth can carve the training data into ever smaller pieces. At some point it stops learning the *pattern* and starts memorising the *examples*. Watch both scores as the depth grows:

In [ ]:
from sklearn.tree import DecisionTreeClassifier

depths = range(1, 16)
train_acc, test_acc = [], []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train, y_train)
    train_acc.append(t.score(X_train, y_train))
    test_acc.append(t.score(X_test, y_test))

plt.plot(depths, train_acc, marker="o", color="#1d3fbf", label="training data (seen)")
plt.plot(depths, test_acc, marker="o", color="#c0392b", label="test data (unseen)")
plt.axhline(baseline, color="gray", linestyle="--", label="always predict 'pass'")
plt.xlabel("Tree depth (complexity)")
plt.ylabel("Accuracy")
plt.title("More complexity: better at memorising, worse at predicting")
plt.legend()
plt.show()

Now take complexity to its limit — a tree with **no depth limit at all**:

In [ ]:
deep = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)

print(f"On the data it trained on: {deep.score(X_train, y_train):.0%}")
print(f"On new learners:           {deep.score(X_test, y_test):.0%}")
print(f"Always predicting 'pass':  {baseline:.0%}")

A perfect score on the training data — and **worse than doing nothing** on learners it has never seen. That is **overfitting**: the model memorised 675 individual learners, noise and all, instead of learning what predicts success.

## Choosing the depth honestly: cross-validation

It is tempting to just pick the depth with the best *test* score from the chart. Don't: the moment the test set helps you choose, it is no longer a surprise exam, and your final score becomes optimistic.

**Cross-validation** chooses using the *training* data only. It splits it into 5 parts, trains on 4 and scores on the 5th — five times, rotating — then averages:

In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = {}
for d in depths:
    tree_d = DecisionTreeClassifier(max_depth=d, random_state=42)
    cv_scores[d] = cross_val_score(tree_d, X_train, y_train, cv=5).mean()

best_depth = max(cv_scores, key=cv_scores.get)
print(f"Best depth by cross-validation: {best_depth} (average accuracy {cv_scores[best_depth]:.1%})")

Cross-validation lands on the same depth Lesson 1 used — chosen this time without ever looking at the test set.

## Comparing models

Depth is one choice; the *type* of model is another. Here are five candidates, each scored by cross-validation on the training data first. *(The random forest trains 100 trees, so this cell takes a few seconds in the browser.)*

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

models = {
    "Always 'pass'": DummyClassifier(strategy="most_frequent"),
    f"Decision tree (depth {best_depth})": DecisionTreeClassifier(max_depth=best_depth, random_state=42),
    "Decision tree (no limit)": DecisionTreeClassifier(random_state=42),
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression()),
    "Random forest": RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
}

results = {}
for name, model in models.items():
    results[name] = cross_val_score(model, X_train, y_train, cv=5).mean()
    model.fit(X_train, y_train)

pd.Series(results, name="cross-validated accuracy").sort_values(ascending=False).round(3)

The unlimited tree sinks to the bottom, as expected. But look at the top three: the depth-limited tree, logistic regression and the random forest are **within about a point of each other**. With 675 training examples, differences that small are noise — on accuracy, it is a tie.

So how do you choose?

## ⚖️ Ethics lab — the price of a better score

**Beat 1 — Build it** ✅ (five fitted models).

**Beat 2 — Measure it.** In Lesson 1, the tree was far less accurate for learners without their own laptop. Does a more powerful model fix that? `fairness_table()` audits every model at once:

In [ ]:
%run datalab_ethics.py

laptop = df.loc[X_test.index, "has_own_laptop"].map({1: "has laptop", 0: "no laptop"})
fairness_table(models, X_test, y_test, by=laptop)

Read the `accuracy: no laptop` column, then compare it with the `Always 'pass'` row.

**For learners without a laptop, every model you trained is *less* accurate than simply predicting "pass" for everyone.** All the accuracy these models gained, they gained on laptop owners. And the most powerful model, the random forest, has the **widest** gap of all — it extracts more from the group the data describes well, and nothing from the group it cannot.

Meanwhile every row passes the four-fifths screen. On the usual checks, all of these models look fine.

**Beat 3 — Name it.** "Pick the most accurate model" sounds neutral. It is not: overall accuracy is an average dominated by the majority, so optimising it quietly decides that the majority's errors matter more. The EU AI Act requires high-risk systems to reach an *appropriate level of accuracy* (Art. 15(1)) and their accuracy metrics to be declared to deployers (Art. 15(3)) — but it does not tell you *whose* accuracy counts. That choice is yours, and it should be made openly.

What a professional would weigh:

- **A simpler model with a smaller gap**, even at a small cost in overall accuracy
- **Reporting accuracy per group**, not just overall, in the model's documentation
- **Human review** for decisions about the group the model serves badly (Art. 14)
- **Better data** about that group — the only option that actually fixes the cause

**Beat 4 — Judge it.** Your call, in writing:

In [ ]:
my_decision = """
Your answer: which model would you deploy to allocate scholarship places, and why?
Name exactly what you are trading off.
"""

reflect_submit("ml-model-choice", my_decision)

## ✅ What you learned

Overfitting and how to see it · why the test set must not choose your model · cross-validation · comparing models against a baseline · that models tied on accuracy can differ sharply on fairness · and that *"the most accurate model"* is a value choice, not a neutral one.

📚 **Go further:**
- [scikit-learn: cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html)
- [Fairlearn: assessing fairness](https://fairlearn.org/main/user_guide/assessment/index.html)

**Next:** the *capstone* — build, audit and document a model of your own, and export your portfolio.